# Try it yourself: Mistral-7B before vs after fine-tuning

This notebook runs my side-by-side demo on Google's free computers, so you can try it without installing anything.

- **A. Original Mistral-7B-Instruct v0.3**: the base model
- **C. My fine-tuned model**: the same base + a QLoRA adapter trained on 4,050 Bitext customer-support conversations ([model page](https://huggingface.co/pramod586/mistral-7b-customer-support-lora))

**How to use**
1. *(Recommended)* **Runtime → Change runtime type → T4 GPU** for fast answers. A CPU runtime also works, but each answer takes 1-2 minutes.
2. **Runtime → Run all**.
3. After about 5-10 minutes the last cell prints a link like `https://xxxx.gradio.live`. Open it and ask a customer-support question.

The link works while this notebook is running. Stop the last cell (■) when you are done.

## 1. Install Ollama (runs the models)

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

## 2. Start the Ollama server in the background

In [ ]:
import subprocess, time, requests

server = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("Ollama server is running")
        break
    except requests.RequestException:
        time.sleep(1)
else:
    print("Ollama did not start - see ollama.log")

## 3. Download both models (about 4.4 GB each)

In [ ]:
# C. My fine-tuned model, straight from my Hugging Face page
!ollama pull hf.co/pramod586/mistral-7b-customer-support-GGUF:Q4_K_M
!ollama cp hf.co/pramod586/mistral-7b-customer-support-GGUF:Q4_K_M support-agent

# A. The original Mistral-7B-Instruct v0.3, same compression (Q4_K_M)
!ollama pull mistral:7b-instruct-v0.3-q4_K_M
!ollama cp mistral:7b-instruct-v0.3-q4_K_M mistral-original

!ollama list

## 4. Get the demo app

In [ ]:
REPO = "https://github.com/pramod-nanu/mistral-7b-customer-support-finetune"
!rm -rf repo && git clone -q {REPO} repo
!pip install -q -r repo/demo/requirements.txt

## 5. Launch the demo
Wait for the line `Running on public URL: https://....gradio.live` and open that link.

In [ ]:
!cd repo && SHARE=1 python demo/app.py